# Progetto DIA - A.A 2025/26

Autori: Justin Carideo (justin.carideo@studio.unibo.it), Laura Bertozzi (laura.bertozzi5@studio.unibo.it)

# Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import seaborn as sns

sns.set_theme(style="whitegrid")

Importiamo i dataset presi dal sito https://www.kaggle.com. Questi due dataset fanno riferimento a dati in ambito agricolo, in particolare:
1) **Crop Dataset** -> dataset incentrato su aspetti ambientali per individuare il tipo di coltura dato il tipo di terreno e nutrienti presenti (https://www.kaggle.com/datasets/snmahsa/soil-nutrients)
2) **Fertilizer DataSet** -> dataset incentrato sullo stesso campo ma che vuole individuare il tipo di fertilizzante dato il tipo di terreno e la coltura presente. (https://www.kaggle.com/datasets/nishchalchandel/fertilizer-recommendation)

In [ ]:
import kaggle
import os
import os.path

CROP_DATASET_ID = "varshitanalluri/crop-recommendation-dataset"
FERTILIZER_DATASET_ID = "nishchalchandel/fertilizer-recommendation"

BASE_DOWNLOAD_DIR = "data"

CROP_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_1")
FERTILIZER_DATASET_DIR = os.path.join(BASE_DOWNLOAD_DIR, "dataset_2")

os.makedirs(BASE_DOWNLOAD_DIR, exist_ok=True) # crea la directory se non esiste


def download_kaggle_dataset(dataset_id, dataset_path):
    os.makedirs(dataset_path, exist_ok=True)
    if not os.listdir(dataset_path):
        print(f"Download {dataset_id} in {dataset_path}")
        kaggle.api.dataset_download_files(
            dataset_id,
            path=dataset_path,
            unzip=True
        )
    else:
        print(f"Dataset already in specified path")

download_kaggle_dataset(CROP_DATASET_ID, CROP_DATASET_DIR)
download_kaggle_dataset(FERTILIZER_DATASET_ID, FERTILIZER_DATASET_DIR)

CROP_DATASET_PATH = os.path.join(CROP_DATASET_DIR, "Crop_recommendation.csv")
FERTILIZER_DATASET_PATH = os.path.join(FERTILIZER_DATASET_DIR, "fertilizer_recommendation_dataset.csv")

Creiamo i dataframe di entrambi i dataset:

In [ ]:
crop_df = pd.read_csv(CROP_DATASET_PATH)
fertilizer_df = pd.read_csv(FERTILIZER_DATASET_PATH)

Ora che abbiamo ottenuto i dataset dobbiamo decidere che modello attuare su questi dataset e visto che non hanno indici univoci per riga possiamo attuare una **recommendation** per la predizione di colture e fertilizzanti. In particolare potremmo ragionare con una sorta di *pipeline* dove i dati di entrambi vengono utilizzati per ottenere una classificazione della coltura desiderata date le informazioni relative al terreno e utilizzare i dati ottenuti per calcolare anche il tipo di fertilizzante da attuare.

Questa è una analisi a priori, una volta svolta l'*analisi esplorativa* si potrà procedere con la costruzione del modello.

# Analisi Esplorativa

In [ ]:
crop_df.head()

In [ ]:
fertilizer_df.head()

In [ ]:
crop_df.info()

In [ ]:
fertilizer_df.info()

In [ ]:
crop_df.describe()

In [ ]:
fertilizer_df.describe()

Osserviamo ora quali colture vengono prese in considerazione dai dataset.

Poniamo quindi le stringhe dei nomi delle colture in minuscolo ed eliminiamo eventuali spazi, per poi cercare corrispondenze tra di esse.

In [ ]:
crops_fertilizer = fertilizer_df["Crop"].unique()
crops_fertilizer = [x.lower() and x.strip() for x in crops_fertilizer]
print(crops_fertilizer)

In [ ]:
crops_crop = crop_df["Crop"].unique()
crops_crop = [x.lower() and x.strip() for x in crops_crop]
print(crops_crop)

In [ ]:
common = list(set(crops_fertilizer) & set(crops_crop))
print("Common elements:", common)
difference = list(set(crops_crop) - set(common))
print("Elements in crop but not in fertilizer:", difference)

print("Number of common elements:", len(common))
print("Number of elements in crop:", len(crops_crop))

Notiamo che le colture di `crop_df` rientrano perfettamente nelle colture di `fertilizer_df`. Di conseguenza possiamo portarle allo stesso formato, poi effettuare un join.

In [ ]:
def clean_crop_name(name):
    return str(name).lower().replace(" ", "")

In [ ]:
crop_df["Crop"] = crop_df["Crop"].map(clean_crop_name)

In [ ]:
fertilizer_df["Crop"] = fertilizer_df["Crop"].map(clean_crop_name)

In [ ]:
common_crops = set(fertilizer_df["Crop"]).intersection(set(crop_df["Crop"]))
print(common_crops)
print(f"Common crops = {len(common_crops)}")

### Osservazione dei campioni mediante diagrammi



Innanzitutto analizziamo le variabili prese in considerazione dai due dataset, ponendo particolare attenzione su quelle in comune.

In [ ]:
crop_keys = set(crop_df.keys())
fertilizer_keys = set(fertilizer_df.keys())


print("variabili di \"crop_df\":", list(crop_keys))
print("numero di variabili di \"crop_df\":", len(crop_keys))
print("\nvariabili di \"fertilizer_df\":", list(fertilizer_keys))
print("numero di variabili di \"fertilizer_df\":", len(fertilizer_keys))

common_keys = list(set(crop_keys) & set(fertilizer_keys))
print("\nvariabili comuni:", common_keys)
print("numero di variabili comuni:", len(common_keys))

different_a = list(set(fertilizer_keys) - set(common_keys))
different_b = list(set(crop_keys) - set(common_keys))
different_keys = list(set(crop_keys) ^ set(fertilizer_keys))
print("\nvariabili diverse:", different_keys)
print("numero di variabili diverse:", len(different_keys))

Tra le variabili diverse notiamo che tre di esse differiscono solo nel nome, ossia:
* Humidiy e Moisture
* PH e pH_Value
* Phosforrous e Phosforus

Le restanti quattro rappresentano parametri effettivamente differenti:
* Soil
* Carbon
* Fertilizer (incognita del fertilizer dataset)
* Remark (che ignoriamo per il momento)

Ora procediamo valutando le unità di misura.

Notiamo come alcuni parametri non facciano riferimento alla stessa scala(i.e. humidity -> in percentuale (72%), Moisture -> in proporzione (0.72)). In compenso, per quanto riguarda il resto:
* Temperature -> °C
* Nitrogen, Phosphorous, Carbon, Potassium -> mg/kg
* pH -> misura adimensionale standard (da 0 a 14)
* Rainfall -> mm

Dopo grafici scrivi:
- Hanno 100 esemplari per coltura

Distribuzione delle Colture:

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=crop_df, x='Crop', hue='Crop', palette='viridis', order=crop_df["Crop"].value_counts().index)
plt.title("Distribuzione Colture (crop_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
sns.countplot(data=fertilizer_df, x='Crop', hue='Crop', palette='viridis', order=fertilizer_df["Crop"].value_counts().index)
plt.title("Distribuzione Colture (fertilizer_df)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

Valori di umidità rilevati:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(crop_df["Humidity"], kde=True, color='blue', ax=axes[0])
axes[0].set_title('Humidity crop_df', fontsize=12)
axes[0].set_xlabel('Humidity (%)')

sns.histplot(fertilizer_df["Moisture"], kde=True, color='red', ax=axes[1])
axes[1].set_title('Humidity fertilizer_df', fontsize=12)
axes[1].set_xlabel('Humidity (ratio)')

Temperature misurate:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(crop_df["Temperature"], kde=True, color='blue', ax=axes[0])
axes[0].set_title('Temperature crop_df', fontsize=12)
axes[0].set_xlabel('Temperature (°C)')

sns.histplot(fertilizer_df["Temperature"], kde=True, color='red', ax=axes[1])
axes[1].set_title('Temperature fertilizer_df', fontsize=12)
axes[1].set_xlabel('Temperature (°C)')

### Mappatura variabili ad un formato comune

In [ ]:
map_names = {
    "Moisture": "Humidity",
    "PH" : "pH_Value",
    "Phosphorous" : "Phosphorus"
}

fertilizer_df = fertilizer_df.rename(columns=map_names)
fertilizer_df["Humidity"] = fertilizer_df["Humidity"] * 100 # Per evitare problemi con la divisione

In [ ]:
fertilizer_df.head()

In [ ]:
crop_df.head()

In [ ]:
column_names = ["Temperature", "Humidity", "Rainfall", "pH_Value", "Nitrogen", "Phosphorus", "Potassium", "Crop"]

merged_df = pd.concat([
    fertilizer_df[column_names],
    crop_df[column_names]],
    axis=0,
    join='outer',
    ignore_index=True
)

merged_df.describe()

# Addestramento modello 1

Una volta estratte le feature più importanti possiamo passare al tipo di modello che vogliamo addestrare.
Il primo modello sarà un modello di *classificazione* per la previsione del parametro `Crop` attraverso degli **alberi di classificazione**, in particolare alleniamo due modelli:
- uno che userà `RandomForestClassifier`, che utilizza *Random Forest*
- uno che userà `xgboost`, che utilizza *Gradient Boosting*